# Positional Encoding - 从零实现到 PyTorch 版本

## 目标：彻底搞懂 Transformer 的位置编码

1. **为什么需要位置编码** —— Self-Attention 的致命缺陷
2. **公式的直觉** —— 为什么是正余弦？为什么是这个频率？
3. **从零实现** —— 纯 numpy 手动推导
4. **PyTorch 实现** —— 可插入真实 Transformer
5. **关键性质验证** —— 唯一性 & 相对位置线性变换
6. **可视化** —— ASCII 热力图 + 交互式频率演示
7. **完整 Mini-Transformer** —— 端到端接入演示


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn

print("所有依赖已加载！")
print(f"numpy version: {np.__version__}")
print(f"torch version: {torch.__version__}")


---

# PART 1: 为什么需要位置编码？—— Self-Attention 的致命缺陷

## 问题场景

考虑这两句话：
- "我 打 你" —— 意思是：我攻击你
- "你 打 我" —— 意思是：你攻击我

**词完全一样，但顺序不同，意思完全相反！**

## Self-Attention 的计算过程

Self-Attention 的核心计算是：

Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V

注意：这里只做了矩阵乘法和 softmax，**完全没有用到位置信息**！

如果 Q=K=V=X，那么交换 X 的行（即交换 token 顺序），结果不变。这就是**排列不变性**（Permutation Invariance）。

对于分类任务（如判断图片类别），排列不变性是好事。但对于序列任务（如翻译），这是**致命缺陷**！


In [ ]:
def simple_self_attention(X):
    """
    极简化 Self-Attention（Q=K=V=X，不带 weight matrix）
    只是为了演示"排列无关性"
    """
    # attention scores: (seq_len, seq_len)
    scores = X @ X.T / math.sqrt(X.shape[1])
    # softmax
    exp_s = np.exp(scores - scores.max(axis=-1, keepdims=True))
    attn = exp_s / exp_s.sum(axis=-1, keepdims=True)
    return attn @ X  # weighted sum

# 假设三个词的 embedding
wo  = np.array([0.1, 0.8, 0.2, 0.5])  # "我"
da  = np.array([0.9, 0.1, 0.7, 0.3])  # "打"
ni  = np.array([0.3, 0.6, 0.1, 0.9])  # "你"

sentence_A = np.stack([wo, da, ni])  # "我打你"
sentence_B = np.stack([ni, da, wo])  # "你打我"

out_A = simple_self_attention(sentence_A)
out_B = simple_self_attention(sentence_B)


In [ ]:
print("=" * 60)
print("句子 A「我打你」的 token 顺序：", ["我", "打", "你"])
print("句子 B「你打我」的 token 顺序：", ["你", "打", "我"])
print("=" * 60)

print("\n句子 A「我打你」的 attention 输出：")
print(np.round(out_A, 4))

print("\n句子 B「你打我」的 attention 输出（顺序不同，但各 token 输出完全一样）：")
print(np.round(out_B, 4))

print('\n' + "=" * 60)
print('结论：out_A[0]（"我"的输出）== out_B[2]（"我"的输出）：',
      np.allclose(out_A[0], out_B[2]))
print("=" * 60)
print("\n问题：没有位置编码时，模型完全感知不到词序！")
print("→ 无论 "我" 在位置 0 还是位置 2，输出完全相同！")


## 解决方案：位置编码

我们需要给每个 token 注入一个**位置信号**，使得：

1. **不同位置有不同的编码** —— 位置 0 ≠ 位置 1 ≠ 位置 2
2. **模型能学到位置之间的关系** —— 例如：位置 5 和位置 6 很接近，位置 5 和位置 50 很遥远

Transformer 论文提出了两种方案：

| 方案 | 方法 | 优缺点 |
|------|------|--------|
| **方案 A** | 可学习的位置编码（Learnable PE） | 简单，但泛化到训练时没见过的长度可能不好 |
| **方案 B** | **固定的正余弦位置编码**（Sinusoidal PE） | 无需训练，可泛化到任意长度，有优美的数学性质 |

论文最终选择了方案 B，这就是我们要深入理解的公式。


---

# PART 2: 公式拆解 —— 为什么是正余弦？为什么是这个频率？

## 公式（原论文）

PE(pos, 2i)   = sin( pos / 10000^(2i / d_model) )

PE(pos, 2i+1) = cos( pos / 10000^(2i / d_model) )

## 变量含义

- `pos` → token 在序列中的位置（0, 1, 2, ..., max_len-1）
- `i` → embedding 维度的「对」索引（0 ~ d_model/2 - 1）
- `d_model` → embedding 总维度（如 512，必须是偶数）

### 核心设计思路：用「多分辨率时钟」编码位置

想象一下你要记录时间：
- **秒针**：每秒跳一格 —— 高频，能精确区分最近的时间点
- **分针**：每分钟跳一格 —— 中频，能区分几分钟内的时间
- **时针**：每小时跳一格 —— 低频，能区分几小时内的时间
- **日期**：每天变一次 —— 更低频，能区分更长的时间跨度

位置编码的设计思路完全一样！每个维度对 (2i, 2i+1) 就是一个「时钟」：
- **高频时钟**（i 小）：能精确区分相邻位置（如位置 0 和 1）
- **低频时钟**（i 大）：能区分更远的位置（如位置 0 和 100）


In [ ]:
# 深入理解频率参数
d_model = 512  # Transformer 默认维度

print("频率参数分析（d_model=512）：\n")

# 选取几个有代表性的 i 值
representative_i = [0, 64, 128, 192, 255]

print(f"{'i':>6} | {'分母 10000^(2i/d_model)':>25} | {'频率 ω_i':>15} | {'含义'}")
print("-" * 80)

for i in representative_i:
    denominator = 10000 ** (2 * i / d_model)
    omega_i = 1.0 / denominator
    
    if i == 0:
        meaning = "秒针（最高频，每个位置都变）"
    elif i < 64:
        meaning = "高频（区分相邻位置）"
    elif i < 128:
        meaning = "中频"
    elif i < 200:
        meaning = "低频"
    else:
        meaning = "时针（最低频，几千位置才变）"
    
    print(f"{i:>6} | {denominator:>25.6f} | {omega_i:>15.10f} | {meaning}")

print("\n" + "="*80)
print("关键观察：分母从 1 到 10000 呈指数增长！")
print("→ 频率从 1.0 到 0.0001 呈指数衰减！")
print("→ 这意味着：不同维度覆盖了**非常宽的频率范围**")


## 为什么选择正余弦函数？（最关键的设计决策）

这是整个位置编码最精妙的地方！选择正余弦函数，不是拍脑袋决定的，而是有**深刻的数学原因**。

### 核心数学性质：相对位置的线性变换

对于任意位置 `pos` 和任意偏移量 `k`，都存在一个只依赖 `k` 的矩阵 M(k)，使得：

PE(pos+k) = M(k) · PE(pos)

这意味着：**相对位置关系可以用线性变换来表达！**

### 为什么这很重要？

Transformer 的注意力分数是 QK^T。如果 Q 和 K 都加上了位置编码，那么：

(Q + PE_q)(K + PE_k)^T = QK^T + Q·PE_k^T + PE_q·K^T + PE_q·PE_k^T

最后一项 PE_q · PE_k^T 就能捕捉**两个 token 之间的相对距离**！

因为 PE(pos_q) 和 PE(pos_k) 的点积只依赖于 |pos_q - pos_k|，而不依赖于它们的绝对位置！

### 怎么证明？（利用三角函数和差公式）

sin(a+b) = sin(a)cos(b) + cos(a)sin(b)

cos(a+b) = cos(a)cos(b) - sin(a)sin(b)

对于位置编码，a = pos · ω_i，b = k · ω_i，所以：

sin((pos+k)·ω_i) = sin(pos·ω_i)cos(k·ω_i) + cos(pos·ω_i)sin(k·ω_i)

cos((pos+k)·ω_i) = cos(pos·ω_i)cos(k·ω_i) - sin(pos·ω_i)sin(k·ω_i)

这正好是一个 2×2 旋转矩阵的作用！


In [ ]:
# 可视化：不同频率下 sin/cos 的变化模式
d_model = 16
positions = np.arange(20)

print("不同频率（i=0,1,2,3）下，位置 0~9 的编码值：\n")

# 打印表头
print(f"{'pos':>4}", end="")
for i in range(4):
    freq = 1.0 / (10000 ** (2 * i / d_model))
    print(f"  dim{2*i}(sin,ω={freq:.6f})  dim{2*i+1}(cos,ω={freq:.6f})", end="")
print()
print("-" * 110)

# 打印每个位置的值
for pos in range(10):
    print(f"{pos:>4}", end="")
    for i in range(4):
        freq = 1.0 / (10000 ** (2 * i / d_model))
        s = math.sin(pos * freq)
        c = math.cos(pos * freq)
        print(f"  {s:+.4f}               {c:+.4f}", end="")
    print()

print("\n" + "=" * 110)
print("观察发现：")
print("  1. i=0（最高频，ω≈1.0）：每个位置值都不同，变化剧烈")
print("  2. i=1（ω≈0.158）：变化较慢，几个位置才重复一次模式")
print("  3. i=2（ω≈0.025）：变化更慢")
print("  4. i=3（ω≈0.004）：几乎不变，需要更长序列才能看到变化")
print("\n  → 高频负责区分近邻位置，低频负责区分远距离位置！")


## 为什么用指数衰减的频率？（而不是线性）

公式中的频率是 ω_i = 1/10000^(2i/d_model)，这是**指数衰减**的。

### 为什么不直接用线性频率？

如果频率是线性的：ω_i = i · Δω
- 频率范围有限，可能无法覆盖长序列
- 低频不够低，高频不够高

### 指数衰减的优势

ω_i = 10000^(-2i/d_model)

- 当 i=0：ω_0 = 1（最高频）
- 当 i=d_model/2：ω_(d_model/2) = 10000^(-1) = 0.0001（最低频）

频率范围跨越了 **4 个数量级**（从 1 到 0.0001），能够：
1. **精确区分相邻位置**（高频）
2. **区分中等距离**（中频）
3. **区分远距离**（低频）

### 为什么是 10000 这个底数？

10000 是一个经验值，论文作者发现这个值在实验中效果最好。
- 如果底数太小（如 2），频率衰减太快，低频不够低
- 如果底数太大（如 1e6），频率衰减太慢，高频不够高
- 10000 恰好能在 d_model=512 时，让频率范围覆盖 4 个数量级


---

# PART 3: 从零实现位置编码（纯 numpy）

现在我们把公式翻译成代码，一步一步实现。


In [ ]:
def positional_encoding_numpy(max_len: int, d_model: int) -> np.ndarray:
    """
    从零实现正余弦位置编码

    Args:
        max_len: 支持的最大序列长度
        d_model: embedding 维度（必须为偶数）

    Returns:
        PE 矩阵，shape = (max_len, d_model)
    """
    assert d_model % 2 == 0, "d_model 必须为偶数"

    # ========== 第一步：构造 pos 列向量 ==========
    # shape: (max_len, 1)
    # 每个位置对应一个值：[[0], [1], [2], ..., [max_len-1]]
    pos = np.arange(max_len).reshape(-1, 1)
    print(f"Step 1: pos shape = {pos.shape}")
    print(f"        pos = {pos.flatten()[:5]}...")

    # ========== 第二步：构造频率向量 ==========
    # 原始公式的分母：10000^(2i/d_model)
    # 取对数计算更稳定：exp(2i/d_model * ln(10000))
    # i 的范围：[0, 1, 2, ..., d_model/2 - 1]
    i = np.arange(d_model // 2)
    div_term = np.exp(i * (-math.log(10000.0) / d_model))
    print(f"\nStep 2: div_term shape = {div_term.shape}")
    print(f"         div_term = {np.round(div_term[:5], 6)}...")

    # ========== 第三步：广播乘法，计算角度 ==========
    # angles shape: (max_len, d_model//2)
    # broadcast: (max_len, 1) × (d_model//2,) → (max_len, d_model//2)
    angles = pos * div_term
    print(f"\nStep 3: angles shape = {angles.shape}")
    print(f"         angles[0] (pos=0) = {np.round(angles[0][:5], 4)}")
    print(f"         angles[1] (pos=1) = {np.round(angles[1][:5], 4)}")

    # ========== 第四步：构造输出矩阵 ==========
    # 偶数列用 sin，奇数列用 cos
    PE = np.zeros((max_len, d_model))
    PE[:, 0::2] = np.sin(angles)  # 所有偶数维度（0, 2, 4, ...）
    PE[:, 1::2] = np.cos(angles)  # 所有奇数维度（1, 3, 5, ...）
    print(f"\nStep 4: PE shape = {PE.shape}")

    return PE


In [ ]:
# 演示
print("=" * 60)
print("生成位置编码矩阵（max_len=50, d_model=16）")
print("=" * 60)

PE_np = positional_encoding_numpy(max_len=50, d_model=16)

print("\n" + "=" * 60)
print("前 5 个位置的编码向量：")
print("=" * 60)

for pos in range(5):
    print(f"pos={pos:2d}: {np.round(PE_np[pos], 4)}")


In [ ]:
# 验证：每个位置的编码都是唯一的
print("\n" + "=" * 60)
print("验证：每个位置拥有唯一的位置指纹")
print("=" * 60)

# 计算 pos=0 到其他位置的 L2 距离
for target_pos in [1, 5, 10, 25, 49]:
    distance = np.linalg.norm(PE_np[0] - PE_np[target_pos])
    print(f"  pos=0 到 pos={target_pos:2d} 的 L2 距离: {distance:.6f}")

print("\n结论：所有距离都不为零，说明每个位置的编码都是唯一的！")
print("→ 位置越远，距离越大（不完全严格，但趋势明显）")


---

# PART 4: PyTorch 实现（可接入真实 Transformer）

现在把 numpy 实现转换成 PyTorch 的 nn.Module，可以直接插入到真实的 Transformer 模型中。


In [ ]:
class PositionalEncoding(nn.Module):
    """
    标准正余弦位置编码（来自 "Attention Is All You Need"）

    使用方式：
        pe_layer = PositionalEncoding(d_model=512, max_len=5000, dropout=0.1)
        x = embedding(tokens)       # shape: (batch, seq_len, d_model)
        x = pe_layer(x)             # 注入位置信息后输出同 shape
    """

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # 构造 PE 矩阵（一次性计算，永久缓存）
        # shape: (max_len, d_model)
        pe = torch.zeros(max_len, d_model)

        # pos: (max_len, 1)
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(1)

        # div_term: (d_model//2,)
        # 等价于 1/10000^(2i/d_model)，用 exp(log) 保证数值稳定
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )

        # 偶数列 → sin，奇数列 → cos
        pe[:, 0::2] = torch.sin(position * div_term)  # (max_len, d_model//2)
        pe[:, 1::2] = torch.cos(position * div_term)  # (max_len, d_model//2)

        # 增加 batch 维度 → (1, max_len, d_model)，方便与 (B, T, D) 广播
        pe = pe.unsqueeze(0)

        # register_buffer：不是参数（不参与梯度），但随模型保存/加载
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Token embedding，shape = (batch_size, seq_len, d_model)
        Returns:
            x + PE[:seq_len]，shape 不变
        """
        # self.pe shape: (1, max_len, d_model)
        # x[:, :seq_len] 自动截取到当前序列长度
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


In [ ]:
# 测试
d_model = 16
batch_size = 2
seq_len = 10

pe_layer = PositionalEncoding(d_model=d_model, max_len=100, dropout=0.0)

# 模拟 token embedding 输出
fake_embedding = torch.randn(batch_size, seq_len, d_model)
output = pe_layer(fake_embedding)

print("=" * 60)
print("PositionalEncoding 测试")
print("=" * 60)

print(f"Input  shape: {fake_embedding.shape}  ← (batch, seq_len, d_model)")
print(f"Output shape: {output.shape}          ← 位置编码后，shape 不变")
print(f"\n缓存的 PE 矩阵 shape: {pe_layer.pe.shape}  ← (1, max_len, d_model)")

print("\n前 3 个位置的 PE 向量（前 8 维）：")
for pos in range(3):
    vals = pe_layer.pe[0, pos, :8].numpy()
    print(f"  pos={pos}: {np.round(vals, 4)}")

print("\nPE 是否参与梯度：", pe_layer.pe.requires_grad)
print("→ register_buffer 确保它不会被 optimizer 更新，但随 state_dict 保存。")


---

# PART 5: 验证最关键的数学性质 —— 相对位置的线性变换

这是正余弦位置编码最核心的数学性质，也是 RoPE（旋转位置编码）的灵感来源。


In [ ]:
def verify_linear_transform(d_model=8, pos=7, k=3):
    """验证 PE(pos+k) 确实等于 M(k) @ PE(pos)"""

    PE = positional_encoding_numpy(max_len=100, d_model=d_model)

    pe_pos   = PE[pos]        # PE(pos)
    pe_pos_k = PE[pos + k]    # PE(pos+k)，这是目标

    # 构造 M(k)：每对 (sin, cos) 维度都有一个 2x2 旋转子块
    M = np.zeros((d_model, d_model))
    for i in range(d_model // 2):
        omega_i = 1.0 / (10000 ** (2 * i / d_model))
        angle = k * omega_i
        cos_k = math.cos(angle)
        sin_k = math.sin(angle)
        
        # 第 2i 行/列：旋转矩阵的第一行
        M[2*i,   2*i]   =  cos_k
        M[2*i,   2*i+1] =  sin_k
        
        # 第 2i+1 行/列：旋转矩阵的第二行
        M[2*i+1, 2*i]   = -sin_k
        M[2*i+1, 2*i+1] =  cos_k

    # 用线性变换预测 PE(pos+k)
    pe_predicted = M @ pe_pos

    # 计算误差
    error = np.max(np.abs(pe_predicted - pe_pos_k))

    print(f"验证 pos={pos}, k={k}（d_model={d_model}）：")
    print(f"  PE(pos)      = {np.round(pe_pos[:6], 5)}")
    print(f"  PE(pos+k)    = {np.round(pe_pos_k[:6], 5)}")
    print(f"  M(k)@PE(pos) = {np.round(pe_predicted[:6], 5)}")
    print(f"  最大误差: {error:.2e}")
    print(f"  {'完全吻合！' if error < 1e-10 else '不吻合'}")

    return error


In [ ]:
print("=" * 60)
print("验证：PE(pos+k) = M(k) @ PE(pos)")
print("=" * 60)

# 测试多个不同的 (pos, k) 组合
errors = []
errors.append(verify_linear_transform(d_model=8, pos=5,  k=3))
print()
errors.append(verify_linear_transform(d_model=8, pos=15, k=7))
print()
errors.append(verify_linear_transform(d_model=8, pos=1,  k=10))

print("\n" + "=" * 60)
print(f"所有测试的最大误差: {max(errors):.2e}")
print("结论：数学性质严格成立！")


## 这意味着什么？

这个性质的重要性怎么强调都不为过！它意味着：

### 1. Transformer 能学习相对位置关系

在注意力分数计算中，PE_q · PE_k^T 只依赖于两个 token 之间的**相对距离**，而不依赖于它们的**绝对位置**。

例如：
- 位置 3 和位置 5 的关系（距离=2）
- 位置 100 和位置 102 的关系（距离=2）

这两种关系是**完全相同**的！模型只需要学习一次「距离为 2 的两个 token 是什么关系」，就能应用到任意位置。

### 2. 更好的泛化能力

训练时模型只见过长度为 512 的序列，但推理时可以处理更长的序列（如 1024、2048）。

因为正余弦函数是**无限延伸**的，对于任意位置都能计算出编码值，不需要重新训练。

### 3. RoPE 的灵感来源

后来的研究者发现，如果直接把旋转矩阵 M(k) 应用到 Q 和 K 上（而不是先加 PE 再计算点积），效果更好。

这就是 **RoPE（旋转位置编码）**，被 LLaMA、GPT-NeoX 等现代大模型广泛使用。


---

# PART 6: ASCII 热力图可视化

通过可视化，我们可以直观地看到不同频率维度的变化模式。


In [ ]:
def ascii_heatmap(matrix, title="", row_label="pos", col_label="dim",
                  width=50, height=20):
    """
    将 2D 矩阵渲染为 ASCII 热力图
    颜色：█ (高) → ▓ ▒ ░ → ' ' (低)
    """
    chars = "█▓▒░ "

    # 降采样到 (height, width)
    rows, cols = matrix.shape
    r_idx = (np.linspace(0, rows-1, height)).astype(int)
    c_idx = (np.linspace(0, cols-1, width)).astype(int)
    sampled = matrix[np.ix_(r_idx, c_idx)]

    # 归一化到 [0, 1]
    mn, mx = sampled.min(), sampled.max()
    norm = (sampled - mn) / (mx - mn + 1e-9)

    print(f"\n{title}")
    print(f"  {col_label} →   (共 {cols} 维，低维=高频，高维=低频)")
    print(f"  " + "─" * (width + 4))
    for i, row_val in enumerate(r_idx):
        bar = "".join(chars[int((1 - norm[i, j]) * (len(chars)-1))]
                      for j in range(width))
        print(f"  {row_label}={row_val:3d} │{bar}│")
    print(f"  " + "─" * (width + 4))
    print(f"  {'←高频（i小）':^{width//2}}{'低频（i大）→':^{width//2}}")

PE_vis = positional_encoding_numpy(max_len=50, d_model=64)
ascii_heatmap(
    PE_vis,
    title="正余弦位置编码热力图 (max_len=50, d_model=64)",
    row_label="pos",
    col_label="dim",
    width=60,
    height=25
)


## 热力图解读

- **纵轴**：token 位置（0 在上，49 在下）
- **横轴**：embedding 维度（左边=低维=高频，右边=高维=低频）

- **左侧（高频区域）**：呈现密集的明暗交替
  - 每个位置都有明显的变化
  - 负责区分相邻位置

- **右侧（低频区域）**：几乎是纯色
  - 需要很长的序列才能看出变化
  - 负责区分远距离位置

- **每一行（每个 pos）**：都是独一无二的"条纹指纹"
  - 高频部分提供精细的位置区分
  - 低频部分提供粗糙的位置区分
  - 组合在一起 → 唯一标识每个位置


---

# PART 7: 完整 Mini-Transformer 演示（把 PE 接入真实模型）

现在我们把位置编码接入一个完整的 Transformer Encoder，看看端到端的效果。


In [ ]:
class MiniTransformerEncoder(nn.Module):
    """
    最小化可运行的 Transformer Encoder
    架构：Embedding → PositionalEncoding → TransformerEncoderLayer × N → Linear
    """

    def __init__(self, vocab_size: int, d_model: int, nhead: int,
                 num_layers: int, max_len: int, num_classes: int):
        super().__init__()

        # 1. Token Embedding：把词 id 映射到向量空间
        self.embedding = nn.Embedding(vocab_size, d_model)

        # 2. Positional Encoding：注入位置信息
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout=0.1)

        # 3. Transformer Encoder（多头注意力 + FFN）
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            batch_first=True   # 期望输入 (B, T, D) 而非 (T, B, D)
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 4. 分类头
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, token_ids: torch.Tensor,
                padding_mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            token_ids:    (batch, seq_len) 的整数 tensor
            padding_mask: (batch, seq_len) bool，True=忽略该位置
        Returns:
            logits: (batch, num_classes)
        """
        # Embedding：(B, T) → (B, T, D)
        x = self.embedding(token_ids) * math.sqrt(self.embedding.embedding_dim)
        # sqrt(d_model) 缩放：防止 embedding 值太小被 PE 的量级淹没

        # 注入位置编码：(B, T, D) → (B, T, D)，shape 不变
        x = self.pos_enc(x)

        # Transformer Encoder
        x = self.transformer(x, src_key_padding_mask=padding_mask)

        # 取 [CLS] token（位置 0）的输出做分类
        cls_output = x[:, 0, :]
        logits = self.classifier(cls_output)
        return logits


In [ ]:
# 运行一次前向传播验证
torch.manual_seed(42)
model = MiniTransformerEncoder(
    vocab_size=1000,
    d_model=32,
    nhead=4,
    num_layers=2,
    max_len=128,
    num_classes=3
)

# 模拟一个 batch：2 条句子，长度 10
token_ids = torch.randint(0, 1000, (2, 10))
logits = model(token_ids)

print("=" * 60)
print("Mini-Transformer Encoder 前向传播")
print("=" * 60)

print(f"\n输入 token_ids shape: {token_ids.shape}  ← (batch=2, seq_len=10)")
print(f"输出 logits    shape: {logits.shape}     ← (batch=2, num_classes=3)")
print(f"\n模型参数量：{sum(p.numel() for p in model.parameters()):,}")


## 数据流图

```
token_ids (2, 10)
     ↓  Embedding × sqrt(d_model)
x    (2, 10, 32)   ← 语义向量，但还不知道自己的位置
     ↓  PositionalEncoding（相加，不改变 shape）
x    (2, 10, 32)   ← 现在每个 token 都携带了位置指纹
     ↓  TransformerEncoder（2 层）
x    (2, 10, 32)   ← 每个 token 都「看过了」整个序列
     ↓  取 x[:, 0, :]（CLS token）
cls  (2, 32)
     ↓  Linear
logits (2, 3)      ← 分类输出
```


---

# 总结：位置编码的设计哲学

## 核心问题

Self-Attention 是**排列不变**的，无法感知词序。我们需要给每个 token 注入**位置信号**。

## 三大设计决策

### 1. 为什么是正余弦函数？

**数学性质决定的！** 三角函数的和差公式保证了：

PE(pos+k) = M(k) · PE(pos)

相对距离 k 可以用与绝对位置无关的线性变换表达。这使得 Transformer 能学习**相对位置关系**，而不是死记硬背绝对位置。

### 2. 为什么用指数衰减的频率？

**为了覆盖不同尺度的位置关系！**

- 高频维度（i 小）→ 区分相邻位置（如位置 0 和 1）
- 低频维度（i 大）→ 区分远距离位置（如位置 0 和 100）

指数衰减使得频率范围跨越 4 个数量级，能够精确区分各种距离的位置。

### 3. 为什么是相加而不是拼接？

- **相加**：不增加维度，参数量不变
- **拼接**：维度翻倍，参数量和计算量都会平方增长

在高维空间中，随机向量近似正交，线性层可以学会解耦语义和位置信息。

## 与可学习位置编码的对比

| 特性 | 正余弦 PE（固定） | 可学习 PE |
|------|-------------------|-----------|
| 参数量 | 0（无需学习） | d_model × max_len |
| 训练开销 | 无 | 需要训练 |
| 外推能力 | 优秀（可处理任意长度） | 差（只能处理训练时见过的长度） |
| 相对位置 | 天然支持 | 需要学习 |

## 进阶方向：RoPE

RoPE（旋转位置编码）是正余弦 PE 的改进版：

- 把相加改成「旋转」
- 把旋转矩阵 M(k) 直接融入 Q/K 的矩阵乘法
- 更好的外推性
- LLaMA、GPT-NeoX 等现代大模型都在用

**本质：把正余弦 PE 的线性变换性质发挥到极致！**


In [ ]:
# 你可以在这里实验不同的参数
# 尝试修改 d_model、max_len 等，观察位置编码的变化

# 示例：自定义参数实验
custom_d_model = 8
custom_max_len = 20

PE_custom = positional_encoding_numpy(max_len=custom_max_len, d_model=custom_d_model)

print(f"自定义实验（d_model={custom_d_model}, max_len={custom_max_len}）：")
print("=" * 60)

# 打印每个位置的编码
for pos in range(custom_max_len):
    print(f"pos={pos:2d}: {np.round(PE_custom[pos], 4)}")
